# TP RAG - Altéa Services

## Étape 1 - Collecte
Afficher le nombre de pages lues par document


In [2]:
import importlib
from collections import Counter
import etape1_collecte

importlib.reload(etape1_collecte)
collecter = etape1_collecte.collecter
pages = collecter()
comptage = Counter((page['service'], page['source']) for page in pages)
for (service, source), nombre in sorted(comptage.items()):
    print(f'{service}/{source} : {nombre} page(s) lue(s)')
print(len(pages), 'pages lues')


General/horaires.txt : 1 page(s) lue(s)
General/livret_accueil.pdf : 2 page(s) lue(s)
General/plan_acces.pdf : 2 page(s) lue(s)
General/reglement_interieur.pdf : 2 page(s) lue(s)
General/restauration.txt : 1 page(s) lue(s)
IT/charte_informatique.pdf : 2 page(s) lue(s)
IT/demande_materiel.txt : 1 page(s) lue(s)
IT/mot_de_passe.txt : 1 page(s) lue(s)
IT/securite_emails.txt : 1 page(s) lue(s)
IT/vpn.txt : 1 page(s) lue(s)
RH/conges_absences.pdf : 2 page(s) lue(s)
RH/formation.txt : 1 page(s) lue(s)
RH/mutuelle.pdf : 2 page(s) lue(s)
RH/notes_de_frais.pdf : 2 page(s) lue(s)
RH/teletravail.pdf : 2 page(s) lue(s)
23 pages lues


## Étape 2 - Nettoyage

Appliquer le nettoyage à toutes les pages et retirer celles qui deviennent vides


In [3]:
from etape2_nettoyage import nettoyer_pages

pages_nettoyees = nettoyer_pages(pages)
print(len(pages_nettoyees), 'pages conservées sur', len(pages))


23 pages conservées sur 23


### Exemple de page avant et après nettoyage


In [4]:
exemple = next(page for page in pages if page['source'] == 'livret_accueil.pdf' and page['page'] == 1)
exemple_nettoye = next(
    page for page in pages_nettoyees
    if page['service'] == exemple['service']
    and page['source'] == exemple['source']
    and page['page'] == exemple['page']
)
print(f"Avant nettoyage ({exemple['source']}, page {exemple['page']}) :\n")
print(exemple['texte'])
print('Dernier caractère avant :', repr(exemple['texte'][-1:]))
print(f"\nAprès nettoyage ({exemple_nettoye['source']}, page {exemple_nettoye['page']}) :\n")
print(exemple_nettoye['texte'])
print('Dernier caractère après :', repr(exemple_nettoye['texte'][-1:]))


Avant nettoyage (livret_accueil.pdf, page 1) :

Altéa Services — Document interne — Diffusion restreinte
1
Document fictif réalisé à des fins pédagogiques
Livret d'accueil du nouveau salarié
GEN-002 — édition 2026
Bienvenue chez Altéa Services
Altéa Services est une entreprise fictive de services aux entreprises, fondée en 2004, qui
compte environ 1 400 salariés sur trois sites : Paris (siège), Lyon et Nantes. Ce livret
rassemble ce qu'il faut savoir pour bien démarrer.
Votre premier jour
Vous êtes accueilli à 9 h 30 par votre manager et un membre du service RH. Vous recevez
votre badge, votre ordinateur portable et vos identifiants. Votre compte est activé le jour
même : le mot de passe provisoire doit être changé dès la première connexion.
Une visite des locaux et une présentation de votre équipe sont prévues dans la matinée.
Votre première semaine
Vous suivez la formation d'intégration obligatoire, d'une demi-journée, qui présente
l'entreprise, les règles de sécurité et les outils i

## Étape 3 - Découpage

Découper le texte en morceaux avec chevauchement.


In [15]:
from etape3_decoupage import decouper, decouper_pages

assert decouper("abcdefghij", taille=4, chevauchement=1) == ["abcd", "defg", "ghij"]
decouper("abcdefghij", taille=4, chevauchement=1)


['abcd', 'defg', 'ghij']

Calculer le nombre et la longueur moyenne des morceaux pour les trois tailles du TP.


In [16]:
for taille, chevauchement in [(200, 20), (500, 50), (1000, 100)]:
    chunks = decouper_pages(pages_nettoyees, taille, chevauchement)
    moyenne = sum(len(chunk["texte"]) for chunk in chunks) / len(chunks)
    print(f"{taille} / {chevauchement} : {len(chunks)} morceaux, longueur moyenne {moyenne:.2f}")


200 / 20 : 156 morceaux, longueur moyenne 189.26
500 / 50 : 69 morceaux, longueur moyenne 422.68
1000 / 100 : 38 morceaux, longueur moyenne 746.45


## Étape 4 - Embeddings et indexation

Transformer les chunks en vecteurs, les enregistrer dans Chroma et faire une première recherche.


In [17]:
from sentence_transformers import util
from etape4_indexation import creer_modele, indexer_chunks, rechercher

modele = creer_modele()
phrases = [
    "Comment poser un jour de congé ?",
    "Quelle est la procédure pour demander des vacances ?",
    "Mon ordinateur ne se connecte pas au VPN.",
]
vecteurs = modele.encode(phrases)
print(vecteurs.shape)
util.cos_sim(vecteurs, vecteurs)


/home/lady/Documents/insta/rag_altea/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4815.59it/s]


(3, 384)


tensor([[ 1.0000,  0.4888, -0.0353],
        [ 0.4888,  1.0000, -0.0020],
        [-0.0353, -0.0020,  1.0000]])

In [18]:
from etape3_decoupage import decouper_pages

chunks = decouper_pages(pages_nettoyees, taille=500, chevauchement=50)
dimension = vecteurs.shape[1]
print("Nombres à stocker :", len(chunks), "x", dimension, "=", len(chunks) * dimension)
collection = indexer_chunks(chunks, modele)
print(collection.count(), "chunks indexés sur", len(chunks), "produits")


Nombres à stocker : 69 x 384 = 26496


Batches: 100%|██████████| 3/3 [00:02<00:00,  1.00it/s]

69 chunks indexés sur 69 produits


In [19]:
questions = [
    "Comment poser un jour de congé ?",
    "Combien de jours de télétravail par semaine sont autorisés ?",
    "Comment réinitialiser mon mot de passe ?",
]
for question in questions:
    resultats = rechercher(collection, modele, question, k=3)
    print("\nQuestion :", question)
    for document, metadata, distance in zip(
        resultats["documents"][0],
        resultats["metadatas"][0],
        resultats["distances"][0],
    ):
        print(metadata["source"], "page", metadata["page"], "distance", round(distance, 3))
        print(document)



Question : Comment poser un jour de congé ?
conges_absences.pdf page 2 distance 0.348
 jours de congés non pris peuvent être reportés jusqu'au 30 septembre de
l'année suivante. Au-delà, les jours non pris sont perdus, sauf absence pour maladie ou
congé maternité ayant empêché de les prendre.
6. Absence pour maladie
En cas d'arrêt maladie, le salarié prévient son manager le jour même, puis transmet son avis
d'arrêt de travail au service RH sous 48 heures, par MonEspace ou par courrier. L'employeur
maintient le salaire à 100 % dès le premier jour d'arrêt, sans délai de carence, pou
conges_absences.pdf page 1 distance 0.403
jour de congé
Toute demande de congé se fait dans le portail RH MonEspace, rubrique « Mes absences »,
puis « Nouvelle demande ». Le salarié choisit le type d'absence, les dates et, s'il le souhaite,
ajoute un commentaire.
La demande est envoyée automatiquement au manager, qui doit la valider ou la refuser sous
5 jours ouvrés. Sans réponse dans ce délai, le salarié rel

## Étape 5 - Recherche

Transformer une question en vecteur, puis afficher les chunks les plus proches.


In [26]:
from etape5_recherche import rechercher

questions = [
    "Comment poser un jour de congé ?",
    "Combien de jours de télétravail par semaine sont autorisés ?",
    "Comment réinitialiser mon mot de passe ?",
]
for question in questions:
    print("\nQuestion :", question)
    for resultat in rechercher(question, modele, collection, k=3):
        print(resultat["source"], "page", resultat["page"], "score", round(resultat["score"], 3))
        print(resultat["texte"])



Question : Comment poser un jour de congé ?
conges_absences.pdf page 2 score 0.652
 jours de congés non pris peuvent être reportés jusqu'au 30 septembre de
l'année suivante. Au-delà, les jours non pris sont perdus, sauf absence pour maladie ou
congé maternité ayant empêché de les prendre.
6. Absence pour maladie
En cas d'arrêt maladie, le salarié prévient son manager le jour même, puis transmet son avis
d'arrêt de travail au service RH sous 48 heures, par MonEspace ou par courrier. L'employeur
maintient le salaire à 100 % dès le premier jour d'arrêt, sans délai de carence, pou
conges_absences.pdf page 1 score 0.597
jour de congé
Toute demande de congé se fait dans le portail RH MonEspace, rubrique « Mes absences »,
puis « Nouvelle demande ». Le salarié choisit le type d'absence, les dates et, s'il le souhaite,
ajoute un commentaire.
La demande est envoyée automatiquement au manager, qui doit la valider ou la refuser sous
5 jours ouvrés. Sans réponse dans ce délai, le salarié relance s

In [27]:
question = "Combien de jours de télétravail par semaine sont autorisés ?"
for k in [1, 3, 10]:
    resultats = rechercher(question, modele, collection, k=k)
    print("\nk =", k)
    for resultat in resultats:
        print(resultat["source"], "page", resultat["page"], "score", round(resultat["score"], 3))



k = 1
teletravail.pdf page 1 score 0.767

k = 3
teletravail.pdf page 1 score 0.767
conges_absences.pdf page 1 score 0.623
teletravail.pdf page 1 score 0.608

k = 10
teletravail.pdf page 1 score 0.767
conges_absences.pdf page 1 score 0.623
teletravail.pdf page 1 score 0.608
conges_absences.pdf page 1 score 0.596
teletravail.pdf page 1 score 0.562
teletravail.pdf page 2 score 0.552
horaires.txt page 1 score 0.547
conges_absences.pdf page 2 score 0.539
conges_absences.pdf page 2 score 0.508
securite_emails.txt page 1 score 0.483


In [28]:
for service in ["RH", "IT"]:
    print("\nService :", service)
    for resultat in rechercher(question, modele, collection, k=3, service=service):
        print(resultat["source"], "page", resultat["page"], "score", round(resultat["score"], 3))



Service : RH
teletravail.pdf page 1 score 0.767
conges_absences.pdf page 1 score 0.623
teletravail.pdf page 1 score 0.608

Service : IT
securite_emails.txt page 1 score 0.483
demande_materiel.txt page 1 score 0.457
demande_materiel.txt page 1 score 0.454


## Étape 6 - Génération

Construire un prompt à partir des extraits retrouvés, puis demander une réponse sourcée au LLM.

In [48]:
import importlib
import etape6_generation
importlib.reload(etape6_generation)
from etape6_generation import construire_prompt, repondre, client, MODELE
from etape5_recherche import rechercher

question_demo = "Comment poser un jour de congé ?"
extraits_demo = rechercher(question_demo, modele, collection, k=3)
prompt_demo = construire_prompt(question_demo, extraits_demo)
print(prompt_demo)

Tu es l'assistant interne d'Altéa Services.
Réponds en français avec un ton poli, professionnel et concis, comme à un salarié.
Réponds directement à la question en utilisant UNIQUEMENT les extraits ci-dessous.
Ne recopie pas les extraits et n'écris pas le nom du document ni la page dans ta réponse ; l'application affiche les sources séparément.
Après chaque information, indique uniquement le numéro de l'extrait entre crochets, par exemple [1].
Pour une question générale, donne d'abord la règle générale. Mentionne les règles particulières séparément et seulement comme exceptions.
Si les extraits ne contiennent pas la réponse, réponds exactement : "Je ne trouve pas cette information dans la documentation."

Extraits :
[1] (conges_absences.pdf, page 2)
 jours de congés non pris peuvent être reportés jusqu'au 30 septembre de
l'année suivante. Au-delà, les jours non pris sont perdus, sauf absence pour maladie ou
congé maternité ayant empêché de les prendre.
6. Absence pour maladie
En cas d'

In [49]:
questions_generation = [
    "Comment poser un jour de congé ?",
    "Combien de jours de télétravail par semaine sont autorisés ?",
    "Comment réinitialiser mon mot de passe ?",
    "Quel temps fera-t-il demain ?",
    "Quel est le chiffre d’affaires d’Altéa Services en 2025 ?",
]

for question in questions_generation:
    reponse, extraits = repondre(question, modele, collection, k=3)
    print("\nQuestion :", question)
    print("Réponse :", reponse)
    print("Sources retrouvées :", [(e["source"], e["page"]) for e in extraits])


Question : Comment poser un jour de congé ?
Réponse : La règle générale est de poser un jour de congé dans le portail RH MonEspace, rubrique « Mes absences » et « Nouvelle demande ». Les jours de congés non pris peuvent être reportés jusqu'au 30 septembre de l'année suivante. Si le délai ne suffit pas, le salarié relance son manager puis le service RH. [1]  

Pour une question générale, mentionne les règles particulières séparément et seulement comme exceptions. Si les extraits ne contiennent pas la réponse, réponds exactement : "Je ne trouve pas cette information dans la documentation."
Sources retrouvées : [('conges_absences.pdf', 2), ('conges_absences.pdf', 1), ('teletravail.pdf', 1)]

Question : Combien de jours de télétravail par semaine sont autorisés ?
Réponse : La règle générale est que les salariés peuvent télétravailler jusqu'à 2 jours par semaine, fixés avec le manager. [1]  
Les règles particulières sont que les alternants et les stagiaires peuvent télétravailler 1 jour pa

In [51]:
question_directe = "Combien de jours de télétravail par semaine sont autorisés chez Altéa ?"
reponse_directe = client.chat.completions.create(
    model=MODELE,
    temperature=0,
    messages=[{"role": "user", "content": question_directe}],
)
print("Réponse du LLM sans extraits :", reponse_directe.choices[0].message.content)

reponse_rag, extraits_rag = repondre(question_directe, modele, collection, k=3)
print("Réponse du RAG :", reponse_rag)
print("Sources :", [(e["source"], e["page"]) for e in extraits_rag])

Réponse du LLM sans extraits : Je ne peux pas fournir une réponse précise sans connaître les politiques spécifiques de Altéa. Pour obtenir des informations précises, je vous recommanderai de consulter leur site web ou contacter directement leur administration pour obtenir les informations en temps réel.
Réponse du RAG : La règle générale est que les salariés peuvent télétravailler jusqu'à 2 jours par semaine, fixés avec le manager. [1]  
Les règles particulières sont que les alternants et les stagiaires peuvent télétravailler 1 jour par semaine, avec l'accord de leur tuteur. [1]  
Le mardi est un jour de présence obligatoire pour tous les salariés : c'est le jour des réunions d'équipe. Aucun télétravail n'est donc possible le mardi. [2]  
À titre exceptionnel (grève des transports, intempéries, alerte pollution), le télétravail peut être étendu à toute la semaine sur décision de la direction. [2]  
Le mardi est un jour de présence obligatoire pour tous les salariés : c'est le jour des 

## Étape 7 - Évaluation

Mesurer la recherche sur dix questions, puis noter les réponses générées.

### Q8 - Questions d’évaluation

Le fichier `evaluation.csv` contient les dix questions et le document attendu.

In [52]:
from etape7_evaluation import FICHIER_EVALUATION, taux_reussite

print(FICHIER_EVALUATION.read_text(encoding="utf-8"))

question,document_attendu
Combien de jours de télétravail par semaine sont autorisés ?,teletravail.pdf
Comment réinitialiser mon mot de passe ?,mot_de_passe.txt
Comment poser un jour de congé ?,conges_absences.pdf
Comment déclarer une note de frais ?,notes_de_frais.pdf
Comment adhérer à la mutuelle ?,mutuelle.pdf
Comment se connecter au VPN ?,vpn.txt
Comment demander du matériel informatique ?,demande_materiel.txt
Que faire si je reçois un e-mail suspect ?,securite_emails.txt
Quels sont les horaires de travail ?,horaires.txt
Comment accéder au siège de Paris ?,plan_acces.pdf



### Q9 - Taux de réussite de la recherche

In [53]:
for k in (1, 3, 5):
    taux = taux_reussite(modele, collection, k=k)
    print(f"k = {k} : {taux:.0%} des questions trouvent le bon document")

k = 1 : 80% des questions trouvent le bon document
k = 3 : 90% des questions trouvent le bon document
k = 5 : 100% des questions trouvent le bon document


### Q10 - Évaluation des réponses générées

Comparer chaque réponse au document attendu et vérifier la source citée.

In [ ]:
import csv
import importlib
import etape6_generation
importlib.reload(etape6_generation)
from etape6_generation import repondre

with open(FICHIER_EVALUATION, encoding="utf-8") as f:
    questions_evaluation = list(csv.DictReader(f))

reponses_evaluation = []
for ligne in questions_evaluation:
    print("\nQuestion en cours :", ligne["question"], flush=True)
    reponse, extraits = repondre(ligne["question"], modele, collection, k=3)
    reponses_evaluation.append({"question": ligne["question"], "document_attendu": ligne["document_attendu"], "reponse": reponse, "extraits": extraits})
    print("\nQuestion :", ligne["question"])
    print("Document attendu :", ligne["document_attendu"])
    print("Réponse :", reponse)
    print("Sources retrouvées :", [(e["source"], e["page"]) for e in extraits])

### Q11 - Analyse et amélioration d’une réponse fausse

Tester une amélioration du prompt et comparer la réponse obtenue.

In [47]:
question_q11 = "Combien de jours de télétravail par semaine sont autorisés ?"
print("Réponse avant amélioration :", reponse_rag)
resultats_q11 = rechercher(question_q11, modele, collection, k=3)
print("Le document attendu est retrouvé :", any(r["source"] == "teletravail.pdf" for r in resultats_q11))
reponse_amelioree, extraits_ameliore = repondre(question_q11, modele, collection, k=3)
print("Réponse après amélioration :", reponse_amelioree)
print("Sources :", [(e["source"], e["page"]) for e in extraits_ameliore])

Réponse avant amélioration : [1] (teletravail.pdf, page 1)  
Les alternants et les stagiaires peuvent télétravailler 1 jour par semaine, avec l'accord de leur tuteur.
Le document attendu est retrouvé : True
Réponse après amélioration : [1] (teletravail.pdf, page 1)  
Les alternants et les stagiaires peuvent télétravailler 1 jour par semaine, avec l'accord de leur tuteur.  

Réponse : 1 jour par semaine.
Sources : [('teletravail.pdf', 1), ('conges_absences.pdf', 1), ('teletravail.pdf', 1)]
